In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression, RFE, SelectKBest
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV, ElasticNetCV
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.preprocessing import LabelEncoder, OneHotEncoder


In [4]:
df1 = pd.read_csv('../../../data/processed/land_dataset_final_v3_noisy_3.csv')


In [5]:
df1.shape

(9060, 234)

In [2]:
df = pd.read_csv('../../../data/processed/100k.csv')
# X = df.drop(['price_per_m2'], axis=1)
# y = df['price_per_m2']

In [3]:
df.shape

(100000, 234)

In [4]:
# X = df.drop(columns=["price_per_m2", "address_locality", "address_subdivision", "price", "longitude", "latitude", "geometry"])
# X = X.drop(columns=["mean_price_per_m2", "max_price_per_m2", "median_price_per_m2", "min_price_per_m2"])
X = df.drop(columns=['price_per_m2','price', 'longitude', 'latitude', 'geometry'])
y = df["price_per_m2"]

In [5]:
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Categorical columns: {categorical_cols}")

Categorical columns: ['address_subdivision', 'address_locality', 'address_line_2', 'h_id']


In [6]:
X_original_categorical = X[categorical_cols].copy()

In [ ]:
label_encoders = {}
X_encoded = X.copy()

for col in categorical_cols:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X[col])
    label_encoders[col] = le
    
    # Create mapping dictionary
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f"Encoding map for {col}: {mapping}")

# Save mappings to file (for future use)
import joblib
joblib.dump(label_encoders, '../../../data/processed/label_encoders_v2_100k.pkl')


Encoding map for address_subdivision: {'Phnom Penh': 0}
Encoding map for address_locality: {'Chamkar Mon': 0, 'Chbar Ampov': 1, 'Chraoy Chongvar': 2, 'Dangkao': 3, 'Doun Penh': 4, 'Mean Chey': 5, 'Praek Pnov': 6, 'Prampir Meakkakra': 7, 'Pur SenChey': 8, 'Russey Keo': 9, 'Saensokh': 10, 'Tuol Kouk': 11}
Encoding map for address_line_2: {'Bak Kaeng': 0, 'Boeng Kak Ti Muoy': 1, 'Boeng Kak Ti Pir': 2, 'Boeng Keng Kang Ti Bei': 3, 'Boeng Keng Kang Ti Muoy': 4, 'Boeng Keng Kang Ti Pir': 5, 'Boeng Proluet': 6, 'Boeng Reang': 7, 'Boeng Salang': 8, 'Boeng Thum': 9, 'Boeng Trabaek': 10, 'Boeng Tumpun': 11, 'Chak Angrae Kraom': 12, 'Chak Angrae Leu': 13, 'Chakto Mukh': 14, 'Chaom Chau': 15, 'Chbar Ampov Ti Pir': 16, 'Cheung Aek': 17, 'Chey Chummeah': 18, 'Chhbar Ampov Ti Muoy': 19, 'Chrang Chamreh Ti Muoy': 20, 'Chrang Chamreh Ti Pir': 21, 'Chrouy Changvar': 22, 'Dangkao': 23, 'Kakab': 24, 'Kamboul': 25, 'Kantaok': 26, 'Kaoh Dach': 27, 'Kbal Kaoh': 28, 'Khmuonh': 29, 'Kilomaetr Lekh Prammuoy': 3

['../../../data/processed/label_encoders_v2.pkl']

In [8]:
X_current = X_encoded.copy()
current_feature_names = X_current.columns.tolist()

In [9]:
var_selector = VarianceThreshold(threshold=0.01)
X_filtered = var_selector.fit_transform(X_current)
current_feature_names = [name for i, name in enumerate(current_feature_names) if var_selector.get_support()[i]]
print(X_filtered, "After VarianceThreshold")

[[  5.  78. 450. ...   0.   0.   0.]
 [  0.  48. 476. ...   0.   0.   0.]
 [ 10.  45. 351. ...   0.   0.   0.]
 ...
 [  6.  59. 596. ...   0.   0.   0.]
 [  8.  26. 653. ...   0.   0.   0.]
 [  6.  73.   1. ...   0.   0.   0.]] After VarianceThreshold


In [10]:
mi_selector = SelectKBest(score_func=mutual_info_regression, k=100)
X_mi = mi_selector.fit_transform(X_filtered, y)
current_feature_names = [name for i, name in enumerate(current_feature_names) if mi_selector.get_support()[i]]
print(X_mi, "After Mutual Info")

[[  5.  78. 450. ...  25.  50. 497.]
 [  0.  48. 476. ...  30.  75. 510.]
 [ 10.  45. 351. ...  30.  72. 502.]
 ...
 [  6.  59. 596. ...   0.   0.   0.]
 [  8.  26. 653. ...   0.   0.   0.]
 [  6.  73.   1. ...   0.   0.   0.]] After Mutual Info


In [11]:
lasso = LassoCV(alphas=np.logspace(-5, 0, 100), cv=5, max_iter=10000, tol=1e-2).fit(X_mi, y)
lasso_mask = (lasso.coef_ != 0)
X_embedded = X_mi[:, lasso_mask]
current_feature_names = [name for i, name in enumerate(current_feature_names) if lasso_mask[i]]
print(X_embedded, "After Lasso")

d:\anaconda\envs\env_v3.10\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 732541645.243211, tolerance: 96470295.9577081
  model = cd_fast.enet_coordinate_descent_gram(
d:\anaconda\envs\env_v3.10\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 1239747184.930001, tolerance: 247835305.21328473
  model = cd_fast.enet_coordinate_descent_gram(
d:\anaconda\envs\env_v3.10\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 322982876.62504005, tolerance: 247835305.21328473
  model = cd_fast.enet_coordinate_descent_gram(
d:\anaconda\envs\env_v3.10\lib\site-packages\sklearn\linear_model\_coordinate_descent.p

[[  5.  78. 450. ...  25.  50. 497.]
 [  0.  48. 476. ...  30.  75. 510.]
 [ 10.  45. 351. ...  30.  72. 502.]
 ...
 [  6.  59. 596. ...   0.   0.   0.]
 [  8.  26. 653. ...   0.   0.   0.]
 [  6.  73.   1. ...   0.   0.   0.]] After Lasso


d:\anaconda\envs\env_v3.10\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.184e+09, tolerance: 2.870e+08
  model = cd_fast.enet_coordinate_descent(


In [12]:
if X_embedded.shape[1] < 5:
    recovery_count = min(20, X_mi.shape[1])
    top_indices = np.argsort(mi_selector.scores_)[-recovery_count:]
    X_embedded = X_mi[:, top_indices]
    current_feature_names = [current_feature_names[i] for i in top_indices]

In [13]:
if X_embedded.shape[1] > 1:
    import lightgbm as lgb
    model = lgb.LGBMRegressor(n_estimators=100).fit(X_embedded, y)
    importance = model.feature_importances_
    keep_mask = (importance > np.median(importance))
    X_nonlinear = X_embedded[:, keep_mask]
    current_feature_names = [name for i, name in enumerate(current_feature_names) if keep_mask[i]]
else:
    X_nonlinear = X_embedded
print(X_nonlinear, "After LightGBM")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004757 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11202
[LightGBM] [Info] Number of data points in the train set: 9060, number of used features: 100
[LightGBM] [Info] Start training from score 1587.580247
[[  5.          78.           5.75151681 ... 579.          50.
  497.        ]
 [  0.          48.           2.95158678 ... 627.          75.
  510.        ]
 [ 10.          45.           6.82275341 ... 609.          72.
  502.        ]
 ...
 [  6.          59.          22.01751773 ...   0.           0.
    0.        ]
 [  8.          26.          16.97704226 ...   0.           0.
    0.        ]
 [  6.          73.          22.96786116 ...   4.           0.
    0.        ]] After LightGBM


In [14]:
if X_nonlinear.shape[1] > 5:
    rfe = RFE(
        estimator=RandomForestRegressor(n_estimators=50),
        n_features_to_select=min(15, X_nonlinear.shape[1] - 1),
        step=0.1
    )
    X_final = rfe.fit_transform(X_nonlinear, y)
    current_feature_names = [name for i, name in enumerate(current_feature_names) if rfe.support_[i]]
    df_final = pd.DataFrame(X_final, columns=current_feature_names)
    print(df_final, "Final Features")
else:
    df_final = pd.DataFrame(X_nonlinear, columns=current_feature_names)
    print("Skipping RFE - insufficient features")

      address_locality  address_line_2  near_Koh_Pich_in_km  \
0                  5.0            78.0             5.751517   
1                  0.0            48.0             2.951587   
2                 10.0            45.0             6.822753   
3                 10.0            45.0             6.075819   
4                  4.0            14.0             0.811035   
...                ...             ...                  ...   
9055               2.0             0.0            16.053352   
9056               2.0            66.0            13.149428   
9057               6.0            59.0            22.017518   
9058               8.0            26.0            16.977042   
9059               6.0            73.0            22.967861   

      near_Bassac_Lane_in_km  near_Royal_Palace_in_km  \
0                   4.003117                 5.059849   
1                   2.223151                 3.797103   
2                   4.929488                 4.774171   
3              

In [15]:
df_final

,address_locality,address_line_2,near_Koh_Pich_in_km,near_Bassac_Lane_in_km,near_Royal_Palace_in_km,near_Phnom_Penh_Airport_in_km,n_secondary_school_in_3km_to_5km,n_primary_school_5km,n_primary_school_in_3km_to_5km,n_university_in_2km_to_3km,n_university_in_3km_to_5km,n_resturant_in_1km,n_super_market_5km,n_super_market_in_1km_to_2km,n_bank_5km
0,5.0,78.0,5.751517,4.003117,5.059849,3.942955,48.0,92.0,47.0,47.0,31.0,0.0,80.0,8.0,148.0
1,0.0,48.0,2.951587,2.223151,3.797103,7.980455,38.0,62.0,37.0,10.0,45.0,0.0,78.0,8.0,158.0
2,10.0,45.0,6.822753,4.929488,4.774171,6.130610,41.0,69.0,45.0,40.0,36.0,1.0,69.0,11.0,160.0
3,10.0,45.0,6.075819,4.153498,3.878483,5.951667,44.0,83.0,45.0,18.0,34.0,1.0,92.0,9.0,161.0
4,4.0,14.0,0.811035,2.915045,2.753486,13.113004,29.0,25.0,23.0,3.0,20.0,0.0,64.0,0.0,105.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9055,2.0,0.0,16.053352,15.755190,15.019705,18.944682,0.0,-0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0
9056,2.0,66.0,13.149428,12.928510,11.026332,13.898293,2.0,1.0,2.0,0.0,1.0,0.0,0.0,0.0,-0.0
9057,6.0,59.0,22.017518,20.040635,20.031367,12.113578,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
9058,8.0,26.0,16.977042,14.973843,16.174411,6.074774,2.0,3.0,2.0,0.0,1.0,0.0,0.0,0.0,3.0


In [16]:
df_combined = pd.concat([df_final, y.reset_index(drop=True)], axis=1)
df_combined.to_csv("../../../data/preprocessed/selected_features_dataset_V2.csv", index=False)